# End-to-end pipeline: seq2seq (LSTM) baseline

In [1]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "preprocessing" / "align.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

## 1. Data

In [2]:
from src.data import TedData

ted = TedData()

## 2. Preprocessing

In [3]:
SRC_LANG, TGT_LANG = "en", "az"
DIRECTION = f"{SRC_LANG}-{TGT_LANG}"

train_pairs = [(s, t) for _, s, t in ted.pairs("train", SRC_LANG, TGT_LANG)][:1000]
dev_pairs = [(s, t) for _, s, t in ted.pairs("dev", SRC_LANG, TGT_LANG)][:100]
test_pairs = [(s, t) for _, s, t in ted.pairs("test", SRC_LANG, TGT_LANG)][:100]
len(train_pairs), len(dev_pairs), len(test_pairs)

(1000, 100, 100)

## 3. Model

In [4]:
from src.model import Vocabulary, build_model, translate_corpus, save_translations

MODEL_NAME, SYSTEM, MAX_LENGTH = "seq2seq", "seq2seq", 20

all_pairs = train_pairs + dev_pairs + test_pairs
src_vocab = Vocabulary([s for s, _ in all_pairs])
tgt_vocab = Vocabulary([t for _, t in all_pairs])

model = build_model(MODEL_NAME, src_vocab, tgt_vocab)
model.fit(pairs=list(train_pairs), epochs=10)

  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:18<02:46, 18.50s/it]

 20%|██        | 2/10 [00:34<02:17, 17.18s/it]

 30%|███       | 3/10 [00:51<01:57, 16.82s/it]

 40%|████      | 4/10 [01:08<01:42, 17.13s/it]

...

 70%|███████   | 7/10 [02:00<00:51, 17.32s/it]

 80%|████████  | 8/10 [02:21<00:37, 18.61s/it]

 90%|█████████ | 9/10 [02:44<00:19, 19.80s/it]

100%|██████████| 10/10 [03:07<00:00, 20.71s/it]

100%|██████████| 10/10 [03:07<00:00, 18.71s/it]

## 4. Evaluation

In [5]:
from src.evaluation import evaluate_direction

hyps = translate_corpus(model, [s for s, _ in test_pairs], max_length=MAX_LENGTH)
out_dir = ROOT / "results" / "translations" / SYSTEM
save_translations(out_dir, DIRECTION, [s for s, _ in test_pairs], [t for _, t in test_pairs], hyps)

scores = {}
for metric in ("bleu", "chrf", "comet"):
    try:
        scores[metric] = evaluate_direction(out_dir, DIRECTION, metric)
    except Exception as e:
        scores[metric] = str(e)
scores

  0%|          | 0/100 [00:00<?, ?it/s]

 19%|█▉        | 19/100 [00:00<00:00, 185.43it/s]

 38%|███▊      | 38/100 [00:00<00:00, 160.43it/s]

 55%|█████▌    | 55/100 [00:00<00:00, 152.56it/s]

 75%|███████▌  | 75/100 [00:00<00:00, 167.43it/s]

 92%|█████████▏| 92/100 [00:00<00:00, 157.93it/s]

100%|██████████| 100/100 [00:00<00:00, 144.91it/s]

{'bleu': {'direction': 'en-az',
  'metric': 'bleu',
  'n_sentences': 100,
  'score': 0.7654573885014611,
  'signature': 'nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0'},
 'chrf': {'direction': 'en-az',
  'metric': 'chrf',
  'n_sentences': 100,
  'score': 10.924551464166928,
  'signature': 'nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0'},
 'comet': "cannot import name '_HashedSeq' from 'functools' (/Users/kknopp/.pyenv/versions/3.14.2/lib/python3.14/functools.py)"}